# BERTopic hyperparameter search

Busca em perguntas textualmente únicas, sem perguntas padrão-ouro, `IRRELEVANT_TOPICS` ou `reduce_outliers`. Cada configuração é executada com três seeds. A seleção registra NPMI, diversidade, outliers, concentração, coesão semântica e estabilidade por ARI.

In [1]:
import ast
import hashlib
import json
import platform
from importlib.metadata import version
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score
from umap import UMAP

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_DIR / 'data/perguntas/relevant_question_extraction_gpt-5-4-mini_high_flex.csv'
OUTPUT_DIR = PROJECT_DIR / 'results/topic_hyperparameter_search'
CACHE_DIR = OUTPUT_DIR / 'cache'
EMBEDDING_MODEL = 'google/embeddinggemma-300m'
SEEDS = [42, 123, 2024]
MIN_ARI = 0.85
MAX_OUTLIER_PCT = 30.0
MAX_TOP5_CONCENTRATION_PCT = 30.0
COHESION_TIE_DELTA = 0.01
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

/scratch/victoria.estanislau/g2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(DATA_PATH)
df = df[df['perguntas'].ne('[]')].copy()
df['perguntas'] = df['perguntas'].apply(lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
docs = df.explode('perguntas')['perguntas'].dropna().astype(str).tolist()
unique_docs = pd.Series(docs).drop_duplicates().tolist()
DOCS_HASH = hashlib.sha256('\n'.join(unique_docs).encode()).hexdigest()

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model.max_seq_length = 512
embeddings_path = CACHE_DIR / f'embeddings_{DOCS_HASH[:16]}.npy'
if embeddings_path.exists():
    embeddings = np.load(embeddings_path)
else:
    inputs = [f'task: clustering | query: {doc}' for doc in unique_docs]
    embeddings = embedding_model.encode(inputs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(embeddings_path, embeddings)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print(f'{len(docs):,} ocorrências; {len(unique_docs):,} perguntas únicas')

Batches: 100%|██████████| 2021/2021 [00:52<00:00, 38.26it/s]


18,607 ocorrências; 16,165 perguntas únicas


In [3]:
BASELINE = {'n_neighbors': 15, 'n_components': 5, 'min_cluster_size': 75, 'min_samples': 15, 'cluster_selection_method': 'eom'}
CONFIGS = [
    {'config_id': 'baseline', **BASELINE},
    {'config_id': 'min_cluster_size_20', **BASELINE, 'min_cluster_size': 20},
    {'config_id': 'min_cluster_size_40', **BASELINE, 'min_cluster_size': 40},
    {'config_id': 'min_cluster_size_50', **BASELINE, 'min_cluster_size': 50},
    {'config_id': 'mcs60', **BASELINE, 'min_cluster_size': 60},
    {'config_id': 'mcs70', **BASELINE, 'min_cluster_size': 70},
    {'config_id': 'min_samples_10', **BASELINE, 'min_samples': 10},
    {'config_id': 'min_samples_20', **BASELINE, 'min_samples': 20},
    {'config_id': 'n_components_10', **BASELINE, 'n_components': 10},
    {'config_id': 'n_neighbors_5', **BASELINE, 'n_neighbors': 5},
    {'config_id': 'n_neighbors_10', **BASELINE, 'n_neighbors': 10},
    {'config_id': 'n_neighbors_30', **BASELINE, 'n_neighbors': 30},
    {'config_id': 'nn30_mcs20', **BASELINE, 'n_neighbors': 30, 'min_cluster_size': 20},
    {'config_id': 'nn30_mcs40', **BASELINE, 'n_neighbors': 30, 'min_cluster_size': 40},
    {'config_id': 'nn20_mcs50', **BASELINE, 'n_neighbors': 20, 'min_cluster_size': 50},
    {'config_id': 'nn25_mcs50', **BASELINE, 'n_neighbors': 25, 'min_cluster_size': 50},
]
pd.DataFrame(CONFIGS)

,config_id,n_neighbors,n_components,min_cluster_size,min_samples,cluster_selection_method
0,baseline,15,5,75,15,eom
1,min_cluster_size_20,15,5,20,15,eom
2,min_cluster_size_40,15,5,40,15,eom
3,min_cluster_size_50,15,5,50,15,eom
4,mcs60,15,5,60,15,eom
5,mcs70,15,5,70,15,eom
6,min_samples_10,15,5,75,10,eom
7,min_samples_20,15,5,75,20,eom
8,n_components_10,15,10,75,15,eom
9,n_neighbors_5,5,5,75,15,eom


In [4]:
def topic_words(model):
    return {topic: [word for word, _ in model.get_topic(topic)[:10]] for topic in model.get_topics() if topic != -1}

def npmi_score(words_by_topic, documents):
    analyzer = CountVectorizer(stop_words=list(stopwords.words('portuguese'))).build_analyzer()
    token_sets = [set(analyzer(doc)) for doc in documents]
    n = len(token_sets)
    values = []
    for words in words_by_topic.values():
        for left, right in combinations(words, 2):
            left_count = sum(left in tokens for tokens in token_sets)
            right_count = sum(right in tokens for tokens in token_sets)
            both = sum(left in tokens and right in tokens for tokens in token_sets)
            if both:
                p_left, p_right, p_both = left_count / n, right_count / n, both / n
                values.append(np.log(p_both / (p_left * p_right)) / -np.log(p_both))
    return float(np.mean(values)) if values else np.nan

def semantic_metrics(labels):
    scores = []
    for topic in sorted(set(labels) - {-1}):
        indices = np.flatnonzero(labels == topic)
        if len(indices) < 2:
            continue
        vectors = embeddings[indices]
        leave_one_out = vectors.sum(axis=0) - vectors
        leave_one_out /= np.linalg.norm(leave_one_out, axis=1, keepdims=True)
        scores.append(np.sum(vectors * leave_one_out, axis=1).mean())
    return np.percentile(scores, 10), np.median(scores)

labels_by_run = {}
runs = []
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, stop_words=list(stopwords.words('portuguese')))
for config in CONFIGS:
    for seed in SEEDS:
        run_key = hashlib.sha256(json.dumps({'config': config, 'seed': seed, 'docs_hash': DOCS_HASH}, sort_keys=True).encode()).hexdigest()[:20]
        labels_path = CACHE_DIR / f'labels_{run_key}.npy'
        umap_model = UMAP(n_neighbors=config['n_neighbors'], n_components=config['n_components'], min_dist=0.0, metric='cosine', random_state=seed)
        hdbscan_model = HDBSCAN(min_cluster_size=config['min_cluster_size'], min_samples=config['min_samples'], metric='euclidean', cluster_selection_method=config['cluster_selection_method'], prediction_data=True)
        model = BERTopic(embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer, ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True), top_n_words=10, verbose=False)
        if labels_path.exists():
            labels = np.load(labels_path)
            reduced = umap_model.fit_transform(embeddings)
            model.fit(unique_docs, embeddings=embeddings, y=labels)
        else:
            labels, _ = model.fit_transform(unique_docs, embeddings)
            labels = np.asarray(labels)
            np.save(labels_path, labels)
        labels_by_run[(config['config_id'], seed)] = labels
        words = topic_words(model)
        clustered = labels != -1
        sizes = pd.Series(labels[clustered]).value_counts()
        cohesion_p10, cohesion_median = semantic_metrics(labels)
        runs.append({**config, 'seed': seed, 'npmi': npmi_score(words, unique_docs), 'topic_diversity': len(set(sum(words.values(), []))) / max(1, 10 * len(words)), 'outlier_pct': (~clustered).mean() * 100, 'top5_cluster_concentration_pct': sizes.head(5).sum() / sizes.sum() * 100, 'documents_in_clusters_over_5pct_pct': sizes[sizes / sizes.sum() > 0.05].sum() / sizes.sum() * 100, 'semantic_cohesion_p10': cohesion_p10, 'semantic_cohesion_median': cohesion_median, 'n_topics': len(sizes)})
        print(config['config_id'], seed, len(sizes))
runs_df = pd.DataFrame(runs)
runs_df.to_csv(OUTPUT_DIR / 'metrics_by_seed.csv', index=False)

/scratch/victoria.estanislau/g2/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


baseline 42 47
baseline 123 45
baseline 2024 53
min_cluster_size_20 42 169
min_cluster_size_20 123 188
min_cluster_size_20 2024 168
min_cluster_size_40 42 100
min_cluster_size_40 123 102
min_cluster_size_40 2024 105
min_cluster_size_50 42 76
min_cluster_size_50 123 82
min_cluster_size_50 2024 77
mcs60 42 69
mcs60 123 62
mcs60 2024 64
mcs70 42 52
mcs70 123 50
mcs70 2024 58
min_samples_10 42 51
min_samples_10 123 46
min_samples_10 2024 55
min_samples_20 42 48
min_samples_20 123 43
min_samples_20 2024 46
n_components_10 42 39
n_components_10 123 42
n_components_10 2024 47
n_neighbors_5 42 53
n_neighbors_5 123 53
n_neighbors_5 2024 51
n_neighbors_10 42 48
n_neighbors_10 123 54
n_neighbors_10 2024 46
n_neighbors_30 42 46
n_neighbors_30 123 47
n_neighbors_30 2024 51
nn30_mcs20 42 157
nn30_mcs20 123 159
nn30_mcs20 2024 157
nn30_mcs40 42 102
nn30_mcs40 123 94
nn30_mcs40 2024 92
nn20_mcs50 42 81
nn20_mcs50 123 78
nn20_mcs50 2024 84
nn25_mcs50 42 80
nn25_mcs50 123 82
nn25_mcs50 2024 75


In [5]:
ari_rows = []
for config in CONFIGS:
    config_id = config['config_id']
    for seed_a, seed_b in combinations(SEEDS, 2):
        labels_a = labels_by_run[(config_id, seed_a)]
        labels_b = labels_by_run[(config_id, seed_b)]
        common = (labels_a != -1) & (labels_b != -1)
        ari_rows.append({'config_id': config_id, 'seed_a': seed_a, 'seed_b': seed_b, 'ari': adjusted_rand_score(labels_a[common], labels_b[common]), 'common_clustered_pct': common.mean() * 100})
ari_df = pd.DataFrame(ari_rows)
ari_df.to_csv(OUTPUT_DIR / 'pairwise_ari.csv', index=False)

parameters = ['config_id', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'cluster_selection_method']
summary = runs_df.groupby(parameters).agg(npmi_mean=('npmi', 'mean'), npmi_std=('npmi', 'std'), topic_diversity_mean=('topic_diversity', 'mean'), outlier_pct_mean=('outlier_pct', 'mean'), top5_cluster_concentration_pct_mean=('top5_cluster_concentration_pct', 'mean'), semantic_cohesion_p10_mean=('semantic_cohesion_p10', 'mean'), n_topics_mean=('n_topics', 'mean'), n_topics_std=('n_topics', 'std')).reset_index()
summary = summary.merge(ari_df.groupby('config_id').ari.mean().reset_index(name='ari_mean'), on='config_id')
summary['eligible'] = (summary.ari_mean >= MIN_ARI) & (summary.outlier_pct_mean <= MAX_OUTLIER_PCT) & (summary.top5_cluster_concentration_pct_mean <= MAX_TOP5_CONCENTRATION_PCT)
eligible = summary[summary.eligible].copy()
best_cohesion = eligible.semantic_cohesion_p10_mean.max()
finalists = eligible[eligible.semantic_cohesion_p10_mean >= best_cohesion - COHESION_TIE_DELTA].sort_values(['npmi_mean', 'topic_diversity_mean', 'top5_cluster_concentration_pct_mean'], ascending=[False, False, True])
selected_config_id = finalists.iloc[0].config_id
pairs = ari_df[ari_df.config_id == selected_config_id]
seed_scores = pd.concat([pairs[['seed_a', 'ari']].rename(columns={'seed_a': 'seed'}), pairs[['seed_b', 'ari']].rename(columns={'seed_b': 'seed'})]).groupby('seed').ari.mean().sort_values(ascending=False)
selected_seed = int(seed_scores.index[0])
summary.to_csv(OUTPUT_DIR / 'configuration_summary.csv', index=False)
record = {'configuration': next(config for config in CONFIGS if config['config_id'] == selected_config_id), 'seed': selected_seed, 'thresholds': {'ari_min': MIN_ARI, 'outlier_pct_max': MAX_OUTLIER_PCT, 'top5_concentration_pct_max': MAX_TOP5_CONCENTRATION_PCT}, 'corpus_hash': DOCS_HASH, 'versions': {package: version(package) for package in ['bertopic', 'hdbscan', 'umap-learn', 'sentence-transformers', 'scikit-learn']}, 'python': platform.python_version()}
(OUTPUT_DIR / 'selected_model.json').write_text(json.dumps(record, indent=2, ensure_ascii=False))
display(summary.sort_values(['eligible', 'semantic_cohesion_p10_mean'], ascending=[False, False]))
record

,config_id,n_neighbors,n_components,min_cluster_size,min_samples,cluster_selection_method,npmi_mean,npmi_std,topic_diversity_mean,outlier_pct_mean,top5_cluster_concentration_pct_mean,semantic_cohesion_p10_mean,n_topics_mean,n_topics_std,ari_mean,eligible
14,nn30_mcs20,30,5,20,15,eom,0.393330,0.004906,0.754163,27.881225,21.602178,0.943065,157.666667,1.154701,0.981324,True
15,nn30_mcs40,30,5,40,15,eom,0.380237,0.004342,0.808140,23.319930,24.574240,0.935962,96.000000,5.291503,0.948576,True
13,nn25_mcs50,25,5,50,15,eom,0.374157,0.002199,0.822913,25.384060,24.459764,0.934103,79.000000,3.605551,0.956127,True
12,nn20_mcs50,20,5,50,15,eom,0.372154,0.001755,0.822412,24.029281,26.391482,0.933745,81.000000,3.000000,0.944875,True
4,min_cluster_size_40,15,5,40,15,eom,0.389233,0.001685,0.788323,22.144551,23.620148,0.933033,102.333333,2.516611,0.955547,True
5,min_cluster_size_50,15,5,50,15,eom,0.371524,0.005058,0.829280,21.705330,25.112965,0.930058,78.333333,3.214550,0.949956,True
1,mcs60,15,5,60,15,eom,0.360195,0.008331,0.836637,21.746572,29.109670,0.928808,65.000000,3.605551,0.896496,True
3,min_cluster_size_20,15,5,20,15,eom,0.409174,0.002556,0.739541,25.301577,18.203688,0.941023,175.000000,11.269428,0.820140,False
10,n_neighbors_30,30,5,75,15,eom,0.330387,0.008164,0.857500,19.412311,38.150948,0.928299,48.000000,2.645751,0.827723,False
8,n_components_10,15,10,75,15,eom,0.342155,0.009666,0.870098,18.344159,43.634432,0.927653,42.666667,4.041452,0.942016,False


{'configuration': {'config_id': 'nn30_mcs20',
  'n_neighbors': 30,
  'n_components': 5,
  'min_cluster_size': 20,
  'min_samples': 15,
  'cluster_selection_method': 'eom'},
 'seed': 2024,
 'thresholds': {'ari_min': 0.85,
  'outlier_pct_max': 30.0,
  'top5_concentration_pct_max': 30.0},
 'corpus_hash': '3142c777ba9ca84fe46fcbccb7869d7d754d43b92cb14a7bbf00b0340e8ed679',
 'versions': {'bertopic': '0.17.3',
  'hdbscan': '0.8.40',
  'umap-learn': '0.5.9.post2',
  'sentence-transformers': '5.1.1',
  'scikit-learn': '1.7.1'},
 'python': '3.10.12'}